In [0]:
billing_reference_tbl = dbutils.widgets.get('billing_reference_tbl')
remits_tbl = dbutils.widgets.get('remits_tbl')
claims_tbl = dbutils.widgets.get('claims_tbl')
remits_details_tbl = dbutils.widgets.get('remits_details_tbl')

In [0]:
spark.sql(
    f"""
    WITH
        max_load_timestamp AS (
            SELECT COALESCE(MAX(Loaded_ts), CAST('1900-01-01' AS TIMESTAMP)) AS max_load_date
            FROM {billing_reference_tbl}
        ),
        remits_claims_cte AS (
            SELECT 
                check_number, 
                payee_name, 
                payee_npi, 
                payer_icn, 
                claim_payment, 
                payer_name, 
                pay_date, 
                id,
                patient_ctl_no,
                trans_id,
                `_modified_ts`
            FROM (
                SELECT *,
                    ROW_NUMBER() OVER (
                        PARTITION BY trans_id 
                        ORDER BY _modified_ts DESC
                    ) as rn
                FROM {remits_tbl}
            ) WHERE rn = 1
        ),
        claims_cte AS (
            SELECT 
                patient_dob,
                patient_ctl_no
            FROM (
                SELECT *,
                    ROW_NUMBER() OVER (
                        PARTITION BY patient_ctl_no 
                        ORDER BY _modified_ts DESC
                    ) as rn
                FROM {claims_tbl}
                CROSS JOIN max_load_timestamp
                WHERE _modified_ts > max_load_timestamp.max_load_date
            ) WHERE rn = 1
        ),
        remits_base_values AS (
            SELECT
                TransID,
                Services_LineNo,
                ANY_VALUE(PatientCtlNo) AS PatientCtlNo,
                ANY_VALUE(CustomGroup) AS CustomGroup,
                ANY_VALUE(CAST(CreateDate AS timestamp)) AS CreateDate,
                ANY_VALUE(PatientID_MemberID) AS PatientID_MemberID,
                ANY_VALUE(PatientFirst) AS PatientFirst,
                ANY_VALUE(PatientLast) AS PatientLast,
                ANY_VALUE(TRY_CAST(ReceivedDate AS DATE)) AS ReceivedDate,
                ANY_VALUE(SubscriberLast) AS SubscriberLast,
                ANY_VALUE(AssignedToID) AS AssignedToID,
                ANY_VALUE(StatusCode) AS StatusCode,
                ANY_VALUE(AdjAmounts) AS AdjAmounts,
                ANY_VALUE(Services_Charge) AS Services_Charge,
                ANY_VALUE(Services_HCPC) AS Services_HCPC,
                ANY_VALUE(Services_RevenueCode) AS Services_RevenueCode,
                ANY_VALUE(TRY_CAST(Services_ServiceStart AS DATE)) AS Services_ServiceStart,
                ANY_VALUE(Services_Units) AS Services_Units,
                ANY_VALUE(Underpayment) AS Underpayment,
                ANY_VALUE(Services_Allowed) AS Services_Allowed,
                ANY_VALUE(UnderpaymentAmount) AS UnderpaymentAmount,
                ANY_VALUE(Services_UnitsPaid) AS Services_UnitsPaid,
                ANY_VALUE(Services_Payment) AS Services_Payment,
                ANY_VALUE(RenderingProvID_NPI) AS RenderingProvID_NPI,
                ANY_VALUE(RenderingProvLast) AS RenderingProvLast,
                ANY_VALUE(
                    CASE 
                        WHEN NULLIF(TRIM(Services_Modifier2), '') IS NULL 
                        THEN Services_Modifier1
                        ELSE CONCAT_WS(':', Services_Modifier1, Services_Modifier2)
                    END
                ) AS Modifiers
            FROM {remits_details_tbl}
            CROSS JOIN max_load_timestamp
            WHERE _modified_ts > max_load_timestamp.max_load_date 
              AND _active_flag = 1
            GROUP BY TransID, Services_LineNo
        ),
        remits_adjustments_deduped AS (
            SELECT DISTINCT
                TransID,
                Services_LineNo,
                Services_Adjustments_GroupCode,
                Services_Adjustments_ReasonCode,
                Services_Adjustments_AdjAmount
            FROM {remits_details_tbl}
            CROSS JOIN max_load_timestamp
            WHERE _modified_ts > max_load_timestamp.max_load_date 
            AND _active_flag = 1
        ),
        remits_remarks AS (
            SELECT
                TransID,
                Services_LineNo,
                CONCAT_WS('|', COLLECT_SET(Services_RemarkCodes_RemarkCode)) AS RemarkCodes_RemarkCode
            FROM {remits_details_tbl}
            CROSS JOIN max_load_timestamp
            WHERE _modified_ts > max_load_timestamp.max_load_date 
            AND _active_flag = 1
            GROUP BY TransID, Services_LineNo
        ),
        remits_adjustments AS (
            SELECT
                TransID,
                Services_LineNo,
                CONCAT_WS('|', COLLECT_SET(
                    CONCAT(Services_Adjustments_GroupCode, ':', Services_Adjustments_ReasonCode)
                )) AS Adjustment_Code,
                SUM(CASE 
                    WHEN Services_Adjustments_GroupCode = 'PR'
                    THEN Services_Adjustments_AdjAmount 
                    ELSE 0 
                END) AS Patient_Resp,
                SUM(CASE 
                    WHEN Services_Adjustments_GroupCode != 'PR'
                    THEN Services_Adjustments_AdjAmount 
                    ELSE 0 
                END) AS Service_Line_Adjustments
            FROM remits_adjustments_deduped
            GROUP BY TransID, Services_LineNo
        ),
        remits_claims_id_cte AS (
            SELECT
                b.*,
                a.Adjustment_Code,
                r.RemarkCodes_RemarkCode,
                a.Patient_Resp,
                a.Service_Line_Adjustments
            FROM remits_base_values b
            INNER JOIN remits_adjustments a
                ON a.TransID = b.TransID
                AND a.Services_LineNo = b.Services_LineNo
            LEFT JOIN remits_remarks r
                ON r.TransID = b.TransID
                AND r.Services_LineNo = b.Services_LineNo
        ),
        remits_with_totals AS (
            SELECT 
                *,
                SUM(Service_Line_Adjustments) OVER (
                    PARTITION BY TransID
                ) AS Total_Adj_Amount
            FROM remits_claims_id_cte
        ),
        raw_logic AS (
            SELECT
                NULL AS Adjudication_Date,
                rc.check_number AS Check_EFT_Number,
                rcid.CustomGroup AS Grouping,
                rcid.CreateDate AS Import_Date,
                rcid.PatientID_MemberID AS Member_ID,
                rcid.PatientCtlNo AS Patient_Ctl_No,
                CONCAT(rcid.PatientFirst, ' ', rcid.PatientLast) AS Patient_Name,
                rc.payee_name AS Payee_Name,
                rc.payee_npi AS Payee_NPI,
                rc.payer_icn AS Payer_ICN,
                rc.payer_name AS Payer_Name,
                TRY_CAST(rc.pay_date AS DATE) AS Payment_Date,
                rcid.ReceivedDate AS Received_Date,
                rcid.SubscriberLast AS Subscriber_Name, 
                NULL AS Note_Date,
                NULL AS Notes,
                NULL AS Appeal_Create_Date,
                NULL AS Appeal_Created,
                rcid.AssignedToID AS Assignment,
                NULL AS Followup,
                rc.id AS ID,
                cl.patient_dob AS Patient_DOB, 
                NULL AS Review_Date,
                NULL AS Reviewed,
                NULL AS Reviewed_Reason,
                NULL AS Capital_Outlier,
                NULL AS Operating_Outlier,
                NULL AS Reimbursement_Defined,
                TRY_CAST(NULLIF(rcid.StatusCode, '') AS BIGINT) AS Status_Code,
                rcid.Total_Adj_Amount AS Contract_Adj, 
                rcid.RenderingProvID_NPI AS Rendering_NPI,
                rcid.RenderingProvLast AS Rendering_Provider,
                rc.claim_payment AS Claim_Payment,
                rcid.Modifiers AS Modifiers,
                rcid.Adjustment_Code AS Adjustment_Code,
                NULL AS Adjustment_Code_Description,
                rcid.Services_Allowed AS Allowed_Amount, 
                rcid.Services_Charge AS Charged_Amount, 
                NULL AS Expected_Reimbursement,
                rcid.Services_HCPC AS HCPCS,
                rcid.Patient_Resp AS Patient_Resp,
                rcid.Services_Payment AS Payment_Amount,
                rcid.RemarkCodes_RemarkCode AS Remit_Remark_Code,
                NULL AS Remit_Remark_Description,
                rcid.Services_RevenueCode AS Revenue_Code,
                rcid.Services_ServiceStart AS Service_End,
                rcid.Service_Line_Adjustments AS Service_Line_Adjustments,
                rcid.Services_ServiceStart AS Service_Start,
                rcid.Underpayment,
                rcid.AdjAmounts AS Underpayment_Amount,
                rcid.Services_Units AS Units,
                rcid.Services_UnitsPaid AS Units_Paid,
                rcid.Services_LineNo as service_line_no,
                current_timestamp() AS Loaded_ts
            FROM remits_with_totals rcid
            LEFT JOIN remits_claims_cte rc ON rcid.TransID = rc.trans_id
            LEFT JOIN claims_cte cl ON cl.patient_ctl_no = rc.patient_ctl_no
        )
    INSERT INTO {billing_reference_tbl}
    SELECT  DISTINCT * FROM raw_logic
    """
)